# SHAP Explainability for Intrusion Detection Models

**SOC Analyst Triage AI Assistant - Model Interpretability**

**⚠️ IMPORTANT:** Run this notebook AFTER training both XGBoost and CNN models.

---

## Why SHAP?

**The Problem:**
- XGBoost and CNN are both high-performing but operate as "black boxes"
- SOC analysts need to understand WHY a model flagged an alert
- Regulatory compliance often requires explainable AI
- Model debugging requires understanding decision patterns

**SHAP Solution:**
- **SH**apley **A**dditive ex**P**lanations - based on game theory
- Provides both global (overall) and local (per-prediction) explanations
- Works with ANY model (tree-based, neural networks, etc.)
- Mathematically rigorous and fair attribution

---

## What We'll Do:

1. **XGBoost Explainability** (TreeExplainer - fast, exact)
   - Global feature importance
   - Individual prediction explanations
   - Feature interactions

2. **CNN Explainability** (DeepExplainer - neural networks)
   - Which input features CNN focuses on
   - Comparison with manual features (XGBoost)
   - Learned vs engineered features

3. **Comparative Analysis**
   - Do both models agree on important features?
   - When do they disagree and why?
   - Justification for ensemble approach

4. **Production-Ready Explanations**
   - Dashboard integration examples
   - Real-time explanation generation
   - Analyst-friendly visualizations

---

**Authors:** Napo Qheku & Kabelo Thesele  
**Supervisor:** Mr. Lekuba Ntoane  
**Date:** March 2026

## 1. Setup and Imports

**Why this setup:**
- SHAP library provides model-agnostic explanations
- We need both trained models (XGBoost, CNN) loaded
- Visualization libraries for analyst-friendly plots

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("\n✅ Google Drive mounted successfully!")

In [ ]:
# Install SHAP (if not already installed)
!pip install shap -q

print("✅ SHAP installed")

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# SHAP for explainability
import shap
shap.initjs()  # Initialize JavaScript visualizations for notebooks

# XGBoost
import xgboost as xgb

# TensorFlow/Keras for CNN
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler

# Configuration
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Paths
DRIVE_BASE = Path("/content/drive/MyDrive/SOC_Project")
PROCESSED_DIR = DRIVE_BASE / "data/processed"
MODEL_DIR = DRIVE_BASE / "models"
SHAP_DIR = MODEL_DIR / "shap_results"
SHAP_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("SHAP EXPLAINABILITY SETUP")
print("=" * 60)
print(f"✅ SHAP version: {shap.__version__}")
print(f"✅ XGBoost version: {xgb.__version__}")
print(f"✅ TensorFlow version: {tf.__version__}")
print(f"\n📁 Results will be saved to: {SHAP_DIR}")
print("=" * 60)

## 2. Load Data and Models

**Why we need this:**
- SHAP needs both the trained model AND the data it was trained on
- We'll use a sample of training data as "background" for SHAP
- Test data for generating explanations on actual predictions

In [ ]:
print("=" * 60)
print("LOADING DATA")
print("=" * 60)

print("\n⏳ Loading preprocessed data...\n")

# Load data (same as used for training)
X_train_full = pd.read_csv(PROCESSED_DIR / "X_train.csv")
y_train_full = pd.read_csv(PROCESSED_DIR / "y_train.csv").squeeze()
X_test = pd.read_csv(PROCESSED_DIR / "X_test.csv")
y_test = pd.read_csv(PROCESSED_DIR / "y_test.csv").squeeze()

print(f"✓ Training data: {X_train_full.shape}")
print(f"✓ Test data: {X_test.shape}")
print(f"✓ Features: {X_train_full.shape[1]}")

# Get feature names (important for interpretability!)
feature_names = X_train_full.columns.tolist()
print(f"\n✓ Feature names extracted: {len(feature_names)} features")

# Sample background data for SHAP
# Why sample? Full dataset too slow for SHAP - 100 samples is standard practice
np.random.seed(RANDOM_SEED)
background_indices = np.random.choice(len(X_train_full), size=100, replace=False)
X_background = X_train_full.iloc[background_indices].copy()

print(f"\n✓ Background sample for SHAP: {X_background.shape}")
print("  (100 samples is standard - balances accuracy vs speed)")

# Sample test data for detailed explanations
# Why? We'll show detailed explanations for a few examples
test_sample_size = 1000
test_indices = np.random.choice(len(X_test), size=test_sample_size, replace=False)
X_test_sample = X_test.iloc[test_indices].copy()
y_test_sample = y_test.iloc[test_indices].copy()

print(f"\n✓ Test sample for explanations: {X_test_sample.shape}")
print("=" * 60)

In [ ]:
print("\n" + "=" * 60)
print("LOADING TRAINED MODELS")
print("=" * 60)

# Load XGBoost model
print("\n⏳ Loading XGBoost model...")
xgb_model = xgb.XGBClassifier()
xgb_model.load_model(MODEL_DIR / "xgboost_optimized.json")
print("✓ XGBoost model loaded")

# Load CNN model
print("\n⏳ Loading CNN model...")
cnn_model = keras.models.load_model(MODEL_DIR / "cnn_best_model.h5")
print("✓ CNN model loaded")

# Prepare data for CNN (needs scaling and reshaping)
print("\n⏳ Preparing CNN data (scaling + reshaping)...")
scaler = StandardScaler()

# Fit on training data
X_train_scaled = scaler.fit_transform(X_train_full)
X_background_scaled = scaler.transform(X_background)
X_test_sample_scaled = scaler.transform(X_test_sample)

# Reshape for CNN (add channel dimension)
X_background_cnn = X_background_scaled.reshape(X_background_scaled.shape[0], X_background_scaled.shape[1], 1)
X_test_sample_cnn = X_test_sample_scaled.reshape(X_test_sample_scaled.shape[0], X_test_sample_scaled.shape[1], 1)

print("✓ CNN data prepared")
print(f"  Background shape: {X_background_cnn.shape}")
print(f"  Test sample shape: {X_test_sample_cnn.shape}")

print("\n✅ All models and data ready for SHAP analysis!")
print("=" * 60)

## 3. XGBoost Explainability with SHAP

**Why TreeExplainer:**
- Designed specifically for tree-based models (XGBoost, Random Forest, etc.)
- Exact calculations (not approximate)
- Fast - can handle thousands of predictions
- Provides feature interactions

**What we'll generate:**
1. Global feature importance (summary plot)
2. Individual prediction explanations (waterfall plots)
3. Feature dependence plots (how features affect predictions)
4. Force plots (visual explanation of single predictions)

In [ ]:
print("\n" + "=" * 60)
print("XGBOOST SHAP ANALYSIS")
print("=" * 60)

print("\n⏳ Creating TreeExplainer for XGBoost...")
print("   This analyzes the tree structure to compute exact SHAP values")

# Create explainer
# TreeExplainer is fast and exact for tree models
explainer_xgb = shap.TreeExplainer(xgb_model)
print("✓ TreeExplainer created")

print("\n⏳ Computing SHAP values for test samples...")
print("   This calculates how much each feature contributed to each prediction")
print(f"   Processing {len(X_test_sample)} samples...")

# Calculate SHAP values
# This is the core computation - can take 1-2 minutes
shap_values_xgb = explainer_xgb.shap_values(X_test_sample)

print(f"\n✅ SHAP values computed!")
print(f"   Shape: {shap_values_xgb.shape}")
print(f"   Interpretation: {shap_values_xgb.shape[0]} predictions × {shap_values_xgb.shape[1]} features")
print("\n   Each value represents how much that feature pushed the prediction")
print("   Positive SHAP = pushes toward 'attack', Negative = pushes toward 'benign'")
print("=" * 60)

### 3.1 Global Feature Importance (XGBoost)

**What this shows:**
- Which features are most important OVERALL across all predictions
- Not just magnitude but also variance (spread)
- Color indicates feature value (red=high, blue=low)

In [ ]:
print("\n" + "=" * 60)
print("GLOBAL FEATURE IMPORTANCE (XGBoost)")
print("=" * 60)

print("\nGenerating summary plot...")
print("This shows the TOP features that drive XGBoost predictions globally")

# Summary plot - THE most important visualization
plt.figure(figsize=(12, 8))
shap.summary_plot(
    shap_values_xgb, 
    X_test_sample,
    feature_names=feature_names,
    max_display=20,  # Show top 20 features
    show=False
)
plt.title('XGBoost: Global Feature Importance (SHAP)', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(SHAP_DIR / 'xgboost_global_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Saved: xgboost_global_importance.png")
print("\n📊 How to read this plot:")
print("   • Features ranked by importance (top to bottom)")
print("   • Each dot is one prediction")
print("   • X-axis: SHAP value (impact on prediction)")
print("   • Color: Feature value (red=high, blue=low)")
print("   • Example: If 'sttl' dots are mostly red on the right,")
print("     high sttl values push predictions toward 'attack'")
print("=" * 60)

In [ ]:
# Bar plot version - cleaner for presentations
print("\nGenerating bar plot version (cleaner for presentations)...")

plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values_xgb,
    X_test_sample,
    feature_names=feature_names,
    plot_type="bar",
    max_display=20,
    show=False
)
plt.title('XGBoost: Top 20 Features by Mean |SHAP|', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(SHAP_DIR / 'xgboost_importance_bar.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: xgboost_importance_bar.png")

In [ ]:
# Extract top features for reporting
print("\n" + "=" * 60)
print("TOP 10 MOST IMPORTANT FEATURES (XGBoost)")
print("=" * 60)

# Calculate mean absolute SHAP value for each feature
feature_importance = np.abs(shap_values_xgb).mean(axis=0)
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Mean_|SHAP|': feature_importance
}).sort_values('Mean_|SHAP|', ascending=False)

print("\n")
print(importance_df.head(10).to_string(index=False))

# Save to CSV
importance_df.to_csv(SHAP_DIR / 'xgboost_feature_importance.csv', index=False)
print("\n✓ Saved: xgboost_feature_importance.csv")
print("=" * 60)

### 3.2 Individual Prediction Explanations (XGBoost)

**Why this matters:**
- SOC analysts need to know WHY a specific alert was flagged
- Waterfall plots show the "story" of a prediction
- Starts from base rate, adds contribution of each feature

**Use case:**
"This alert was flagged as malicious because..."
- Feature X increased suspicion by Y
- Feature Z decreased suspicion by W
- Net result: Attack probability 95%

In [ ]:
print("\n" + "=" * 60)
print("INDIVIDUAL PREDICTION EXPLANATIONS (XGBoost)")
print("=" * 60)

# Find interesting examples to explain
# We want: 1) True Positive, 2) False Positive, 3) False Negative

# Get predictions
y_pred_xgb = xgb_model.predict(X_test_sample)
y_pred_proba_xgb = xgb_model.predict_proba(X_test_sample)[:, 1]

# Find examples
tp_idx = np.where((y_test_sample == 1) & (y_pred_xgb == 1))[0][0]  # True Positive
fp_idx = np.where((y_test_sample == 0) & (y_pred_xgb == 1))[0][0]  # False Positive
fn_indices = np.where((y_test_sample == 1) & (y_pred_xgb == 0))[0]
fn_idx = fn_indices[0] if len(fn_indices) > 0 else None  # False Negative (if exists)

print(f"\nFound example predictions:")
print(f"  True Positive (Attack correctly detected): Index {tp_idx}")
print(f"  False Positive (Benign wrongly flagged): Index {fp_idx}")
if fn_idx is not None:
    print(f"  False Negative (Attack missed): Index {fn_idx}")
else:
    print(f"  False Negative: None in sample (excellent recall!)")

print("\n" + "=" * 60)

In [ ]:
# Waterfall plot for True Positive
print("\nExample 1: TRUE POSITIVE (Attack Correctly Detected)")
print(f"Actual: Attack | Predicted: Attack (Probability: {y_pred_proba_xgb[tp_idx]:.2%})")

shap.plots.waterfall(
    shap.Explanation(
        values=shap_values_xgb[tp_idx],
        base_values=explainer_xgb.expected_value,
        data=X_test_sample.iloc[tp_idx],
        feature_names=feature_names
    ),
    max_display=15,
    show=False
)
plt.title('XGBoost Explanation: True Positive (Correctly Detected Attack)', 
         fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(SHAP_DIR / 'xgboost_explanation_true_positive.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Saved: xgboost_explanation_true_positive.png")
print("\n📊 How to read waterfall plot:")
print("   • Starts at E[f(X)] = base rate (expected value)")
print("   • Each bar shows one feature's contribution")
print("   • Red bars push toward 'attack' (increase score)")
print("   • Blue bars push toward 'benign' (decrease score)")
print("   • Final value = f(x) = actual prediction")
print("   • This tells the 'story' of how the model reached its decision")

In [ ]:
# Waterfall plot for False Positive
print("\n" + "=" * 60)
print("Example 2: FALSE POSITIVE (Benign Wrongly Flagged as Attack)")
print(f"Actual: Benign | Predicted: Attack (Probability: {y_pred_proba_xgb[fp_idx]:.2%})")
print("This helps us understand WHY the model made a mistake")

shap.plots.waterfall(
    shap.Explanation(
        values=shap_values_xgb[fp_idx],
        base_values=explainer_xgb.expected_value,
        data=X_test_sample.iloc[fp_idx],
        feature_names=feature_names
    ),
    max_display=15,
    show=False
)
plt.title('XGBoost Explanation: False Positive (Benign Wrongly Flagged)', 
         fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(SHAP_DIR / 'xgboost_explanation_false_positive.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Saved: xgboost_explanation_false_positive.png")
print("\n💡 Why this is valuable:")
print("   • Helps identify features that cause false alarms")
print("   • Analysts can verify if features are suspicious")
print("   • Guides model improvement (retrain with better features)")

In [ ]:
# Waterfall plot for False Negative (if exists)
if fn_idx is not None:
    print("\n" + "=" * 60)
    print("Example 3: FALSE NEGATIVE (Attack Missed)")
    print(f"Actual: Attack | Predicted: Benign (Probability: {y_pred_proba_xgb[fn_idx]:.2%})")
    print("This is CRITICAL - shows why we missed an attack")
    
    shap.plots.waterfall(
        shap.Explanation(
            values=shap_values_xgb[fn_idx],
            base_values=explainer_xgb.expected_value,
            data=X_test_sample.iloc[fn_idx],
            feature_names=feature_names
        ),
        max_display=15,
        show=False
    )
    plt.title('XGBoost Explanation: False Negative (Missed Attack)', 
             fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(SHAP_DIR / 'xgboost_explanation_false_negative.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n✓ Saved: xgboost_explanation_false_negative.png")
    print("\n⚠️ Security implication:")
    print("   • False negatives allow attacks through")
    print("   • Understanding WHY helps prevent future misses")
    print("   • May indicate need for new features or ensemble")
else:
    print("\n✅ No false negatives in sample - excellent recall!")
    print("   This confirms XGBoost's strong attack detection capability")

### 3.3 Feature Dependence Plots (XGBoost)

**What this shows:**
- How a single feature's value affects predictions
- Reveals non-linear relationships
- Shows interaction effects with other features

**Example:**
"High TTL values strongly indicate attacks, but only when connection state is abnormal"

In [ ]:
print("\n" + "=" * 60)
print("FEATURE DEPENDENCE ANALYSIS (XGBoost)")
print("=" * 60)

# Get top 3 features for dependence plots
top_features = importance_df.head(3)['Feature'].tolist()

print(f"\nGenerating dependence plots for top 3 features:")
for i, feat in enumerate(top_features, 1):
    print(f"   {i}. {feat}")

# Create dependence plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, feature in enumerate(top_features):
    shap.dependence_plot(
        feature,
        shap_values_xgb,
        X_test_sample,
        feature_names=feature_names,
        ax=axes[idx],
        show=False
    )
    axes[idx].set_title(f'Dependence: {feature}', fontweight='bold')

plt.suptitle('XGBoost: Feature Dependence Plots (Top 3 Features)', 
            fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(SHAP_DIR / 'xgboost_dependence_plots.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Saved: xgboost_dependence_plots.png")
print("\n📊 How to read dependence plots:")
print("   • X-axis: Feature value")
print("   • Y-axis: SHAP value (impact on prediction)")
print("   • Color: Another feature that interacts with this one")
print("   • Reveals: 'How does changing this feature affect predictions?'")
print("=" * 60)

## 4. CNN Explainability with SHAP

**Why DeepExplainer:**
- Designed for neural networks (CNNs, RNNs, etc.)
- Uses DeepLIFT algorithm (Deep Learning Important FeaTures)
- Approximates SHAP values efficiently for deep models

**Key Difference from XGBoost:**
- CNN learns features automatically from raw inputs
- SHAP shows which INPUT features CNN focuses on
- Interesting to compare with XGBoost's manual features

**Challenge:**
- CNNs are harder to explain than tree models
- SHAP helps but still approximate
- Computation slower (expect 5-10 minutes)

In [ ]:
print("\n" + "=" * 60)
print("CNN SHAP ANALYSIS")
print("=" * 60)

print("\n⏳ Creating DeepExplainer for CNN...")
print("   This uses DeepLIFT to explain neural network predictions")
print("   Background: 100 samples (more = slower but more accurate)")

# Create explainer with background data
# DeepExplainer needs background to compare against
explainer_cnn = shap.DeepExplainer(cnn_model, X_background_cnn)
print("✓ DeepExplainer created")

print("\n⏳ Computing SHAP values for CNN...")
print("   ⚠️ This will take 5-10 minutes (neural networks are complex!)")
print(f"   Processing {len(X_test_sample_cnn)} test samples...")

# Calculate SHAP values
# This is computationally expensive for neural networks
shap_values_cnn = explainer_cnn.shap_values(X_test_sample_cnn)

# DeepExplainer returns list (one per output class)
# For binary classification, index [0] is the SHAP values
if isinstance(shap_values_cnn, list):
    shap_values_cnn = shap_values_cnn[0]

# Remove channel dimension (squeeze from shape (n, 194, 1) to (n, 194))
shap_values_cnn = shap_values_cnn.squeeze()

print(f"\n✅ CNN SHAP values computed!")
print(f"   Shape: {shap_values_cnn.shape}")
print(f"   Interpretation: Shows which INPUT features CNN focuses on")
print("=" * 60)

### 4.1 Global Feature Importance (CNN)

**Comparison with XGBoost:**
- XGBoost uses manual features (engineered by humans)
- CNN learns features automatically from raw data
- Do they agree on what's important?
- Disagreement might reveal complementary strengths!

In [ ]:
print("\n" + "=" * 60)
print("GLOBAL FEATURE IMPORTANCE (CNN)")
print("=" * 60)

print("\nGenerating CNN summary plot...")
print("This shows which INPUT features CNN learns to focus on")

# Convert to 2D array for summary plot
X_test_sample_2d = X_test_sample.values

# Summary plot
plt.figure(figsize=(12, 8))
shap.summary_plot(
    shap_values_cnn,
    X_test_sample_2d,
    feature_names=feature_names,
    max_display=20,
    show=False
)
plt.title('CNN: Global Feature Importance (SHAP)', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(SHAP_DIR / 'cnn_global_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Saved: cnn_global_importance.png")
print("\n💡 Key insight:")
print("   Compare this with XGBoost importance to see if models agree")
print("   Different priorities = complementary strengths = better ensemble!")
print("=" * 60)

In [ ]:
# Bar plot version
print("\nGenerating CNN bar plot...")

plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values_cnn,
    X_test_sample_2d,
    feature_names=feature_names,
    plot_type="bar",
    max_display=20,
    show=False
)
plt.title('CNN: Top 20 Features by Mean |SHAP|', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(SHAP_DIR / 'cnn_importance_bar.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: cnn_importance_bar.png")

In [ ]:
# Extract CNN top features
print("\n" + "=" * 60)
print("TOP 10 MOST IMPORTANT FEATURES (CNN)")
print("=" * 60)

# Calculate mean absolute SHAP value for each feature
feature_importance_cnn = np.abs(shap_values_cnn).mean(axis=0)
importance_df_cnn = pd.DataFrame({
    'Feature': feature_names,
    'Mean_|SHAP|': feature_importance_cnn
}).sort_values('Mean_|SHAP|', ascending=False)

print("\n")
print(importance_df_cnn.head(10).to_string(index=False))

# Save to CSV
importance_df_cnn.to_csv(SHAP_DIR / 'cnn_feature_importance.csv', index=False)
print("\n✓ Saved: cnn_feature_importance.csv")
print("=" * 60)

## 5. XGBoost vs CNN: Feature Importance Comparison

**Critical Question:**
Do XGBoost and CNN agree on what features are important?

**Possible outcomes:**
1. **High agreement** → Features are robustly important
2. **Low agreement** → Models use different strategies (good for ensemble!)
3. **Partial agreement** → Core features + model-specific features

**Why this matters:**
- Validates feature engineering (if models agree)
- Justifies ensemble (if models disagree but both work)
- Guides future feature selection

In [ ]:
print("\n" + "=" * 60)
print("XGBOOST vs CNN: FEATURE IMPORTANCE COMPARISON")
print("=" * 60)

# Merge importance scores
comparison_df = importance_df.merge(
    importance_df_cnn,
    on='Feature',
    suffixes=('_XGB', '_CNN')
)

# Normalize to [0, 1] for fair comparison
comparison_df['XGB_Normalized'] = comparison_df['Mean_|SHAP|_XGB'] / comparison_df['Mean_|SHAP|_XGB'].max()
comparison_df['CNN_Normalized'] = comparison_df['Mean_|SHAP|_CNN'] / comparison_df['Mean_|SHAP|_CNN'].max()

# Calculate agreement (correlation)
from scipy.stats import spearmanr, pearsonr

pearson_corr, pearson_p = pearsonr(comparison_df['XGB_Normalized'], comparison_df['CNN_Normalized'])
spearman_corr, spearman_p = spearmanr(comparison_df['XGB_Normalized'], comparison_df['CNN_Normalized'])

print("\n📊 Feature Importance Correlation:")
print(f"   Pearson correlation:  {pearson_corr:.3f} (p={pearson_p:.4f})")
print(f"   Spearman correlation: {spearman_corr:.3f} (p={spearman_p:.4f})")

if spearman_corr > 0.7:
    print("\n✅ HIGH AGREEMENT: Models prioritize similar features")
    print("   → Features are robustly important")
    print("   → Ensemble may have diminishing returns")
elif spearman_corr < 0.3:
    print("\n⭐ LOW AGREEMENT: Models use different strategies")
    print("   → Complementary approaches")
    print("   → Ensemble highly recommended!")
    print("   → Models catch different attack patterns")
else:
    print("\n💡 MODERATE AGREEMENT: Partial overlap")
    print("   → Core features agreed upon")
    print("   → Each model has unique strengths")
    print("   → Ensemble will likely improve performance")

# Save comparison
comparison_df.to_csv(SHAP_DIR / 'xgb_vs_cnn_feature_comparison.csv', index=False)
print("\n✓ Saved: xgb_vs_cnn_feature_comparison.csv")
print("=" * 60)

In [ ]:
# Visualization: Side-by-side comparison
print("\nGenerating comparison visualization...")

# Get top 15 features from EITHER model
top15_xgb = set(comparison_df.nlargest(15, 'XGB_Normalized')['Feature'])
top15_cnn = set(comparison_df.nlargest(15, 'CNN_Normalized')['Feature'])
top_features_union = list(top15_xgb.union(top15_cnn))

# Filter comparison
plot_df = comparison_df[comparison_df['Feature'].isin(top_features_union)].copy()
plot_df = plot_df.sort_values('XGB_Normalized', ascending=True)

# Create plot
fig, ax = plt.subplots(figsize=(12, 10))

y_pos = np.arange(len(plot_df))
width = 0.35

ax.barh(y_pos - width/2, plot_df['XGB_Normalized'], width, 
        label='XGBoost', color='steelblue', alpha=0.8)
ax.barh(y_pos + width/2, plot_df['CNN_Normalized'], width,
        label='CNN', color='coral', alpha=0.8)

ax.set_yticks(y_pos)
ax.set_yticklabels(plot_df['Feature'])
ax.set_xlabel('Normalized Importance', fontweight='bold')
ax.set_title('XGBoost vs CNN: Feature Importance Comparison (Top Features)', 
            fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(SHAP_DIR / 'xgb_vs_cnn_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Saved: xgb_vs_cnn_comparison.png")
print("\n📊 Interpretation:")
print("   • Features where bars are similar → Both models agree")
print("   • Features where only one bar is high → Model-specific importance")
print("   • Ensemble can leverage both model's strengths!")

In [ ]:
# Agreement vs Disagreement analysis
print("\n" + "=" * 60)
print("AGREEMENT vs DISAGREEMENT ANALYSIS")
print("=" * 60)

# Calculate difference in importance
comparison_df['Importance_Diff'] = abs(comparison_df['XGB_Normalized'] - comparison_df['CNN_Normalized'])

# High agreement features (both models agree they're important)
high_agreement = comparison_df[
    (comparison_df['XGB_Normalized'] > 0.3) & 
    (comparison_df['CNN_Normalized'] > 0.3) &
    (comparison_df['Importance_Diff'] < 0.3)
].sort_values('XGB_Normalized', ascending=False)

print("\n✅ HIGH AGREEMENT FEATURES (Both models prioritize these):")
if len(high_agreement) > 0:
    print(high_agreement[['Feature', 'XGB_Normalized', 'CNN_Normalized']].head(10).to_string(index=False))
    print("\n→ These are robustly important features across different model types")
else:
    print("   None found - models have very different priorities!")

# XGBoost-specific features
xgb_specific = comparison_df[
    (comparison_df['XGB_Normalized'] > 0.3) & 
    (comparison_df['CNN_Normalized'] < 0.2)
].sort_values('XGB_Normalized', ascending=False)

print("\n🔵 XGBoost-SPECIFIC FEATURES:")
if len(xgb_specific) > 0:
    print(xgb_specific[['Feature', 'XGB_Normalized', 'CNN_Normalized']].head(5).to_string(index=False))
    print("\n→ XGBoost finds these important, CNN doesn't")
else:
    print("   None found")

# CNN-specific features
cnn_specific = comparison_df[
    (comparison_df['CNN_Normalized'] > 0.3) & 
    (comparison_df['XGB_Normalized'] < 0.2)
].sort_values('CNN_Normalized', ascending=False)

print("\n🔴 CNN-SPECIFIC FEATURES:")
if len(cnn_specific) > 0:
    print(cnn_specific[['Feature', 'XGB_Normalized', 'CNN_Normalized']].head(5).to_string(index=False))
    print("\n→ CNN finds these important, XGBoost doesn't")
else:
    print("   None found")

print("\n" + "=" * 60)
print("💡 ENSEMBLE IMPLICATION:")
print("=" * 60)
if len(xgb_specific) > 0 or len(cnn_specific) > 0:
    print("\n✅ Models have complementary feature priorities")
    print("   → XGBoost catches attacks through its specific features")
    print("   → CNN catches attacks through its specific features")
    print("   → Ensemble combines both = better overall detection!")
    print("\n📈 Expected ensemble benefit: 1-3% F1-score improvement")
else:
    print("\n⚠️ Models have very similar priorities")
    print("   → Ensemble may have limited benefit")
    print("   → But still worth trying - different algorithms can help")
print("=" * 60)

## 6. Production Use Cases

**How SHAP Enhances Dashboard:**

1. **Alert Explanation Widget**
   ```
   Alert #12345 - FLAGGED AS MALICIOUS
   Confidence: 87%
   
   Why this was flagged:
   + High TTL value (+0.42)
   + Abnormal connection state (+0.31)
   + Suspicious port combination (+0.18)
   - Normal packet count (-0.06)
   ```

2. **Analyst Verification**
   - "Do these features match my domain knowledge?"
   - "Is this a legitimate reason to flag?"
   - "Should I investigate or dismiss?"

3. **Model Debugging**
   - Identify systematic errors
   - Find features causing false positives
   - Guide retraining with better data

In [ ]:
print("\n" + "=" * 60)
print("PRODUCTION USE CASE: REAL-TIME EXPLANATION")
print("=" * 60)

# Simulated dashboard function
def explain_prediction(model_type, sample_idx, show_plot=True):
    """
    Generate explanation for a single prediction.
    This is what the dashboard would call for each alert.
    
    Args:
        model_type: 'xgboost' or 'cnn'
        sample_idx: Index of sample to explain
        show_plot: Whether to show waterfall plot
    
    Returns:
        dict with explanation details
    """
    if model_type == 'xgboost':
        prediction = xgb_model.predict_proba(X_test_sample.iloc[[sample_idx]])[0, 1]
        shap_vals = shap_values_xgb[sample_idx]
    else:  # CNN
        prediction = cnn_model.predict(X_test_sample_cnn[[sample_idx]], verbose=0)[0, 0]
        shap_vals = shap_values_cnn[sample_idx]
    
    # Get top contributing features
    top_pos = []
    top_neg = []
    
    for i, (feat, val) in enumerate(zip(feature_names, shap_vals)):
        if val > 0:
            top_pos.append((feat, val))
        elif val < 0:
            top_neg.append((feat, val))
    
    top_pos = sorted(top_pos, key=lambda x: x[1], reverse=True)[:5]
    top_neg = sorted(top_neg, key=lambda x: x[1])[:5]
    
    # Format for dashboard
    explanation = {
        'prediction': float(prediction),
        'classification': 'ATTACK' if prediction > 0.5 else 'BENIGN',
        'confidence': abs(prediction - 0.5) * 2,  # 0-1 scale
        'top_attack_indicators': top_pos,
        'top_benign_indicators': top_neg
    }
    
    return explanation

# Demo: Explain a specific alert
demo_idx = tp_idx  # Use our True Positive example

print("\nExample Alert Explanation (XGBoost):")
print("="*60)
explanation = explain_prediction('xgboost', demo_idx, show_plot=False)

print(f"\nALERT #{demo_idx}")
print(f"Classification: {explanation['classification']}")
print(f"Probability: {explanation['prediction']:.1%}")
print(f"Confidence: {explanation['confidence']:.1%}")

print("\nTOP ATTACK INDICATORS:")
for feat, val in explanation['top_attack_indicators']:
    print(f"  + {feat}: {val:+.3f}")

print("\nTOP BENIGN INDICATORS:")
for feat, val in explanation['top_benign_indicators']:
    print(f"  - {feat}: {val:+.3f}")

print("\n" + "="*60)
print("💡 This format is perfect for dashboard display!")
print("   Analysts can quickly understand WHY an alert was flagged")
print("="*60)

## 7. Summary and Key Insights

In [ ]:
print("\n" + "=" * 70)
print("SHAP EXPLAINABILITY - SUMMARY")
print("=" * 70)

print("\n✅ COMPLETED ANALYSES:")
analyses = [
    "XGBoost global feature importance (TreeExplainer)",
    "CNN global feature importance (DeepExplainer)",
    "Individual prediction explanations (waterfall plots)",
    "Feature dependence analysis",
    "XGBoost vs CNN comparison",
    "Agreement/disagreement analysis",
    "Production-ready explanation function"
]

for i, analysis in enumerate(analyses, 1):
    print(f"   {i}. {analysis}")

print("\n📁 FILES GENERATED:")
files = [
    "xgboost_global_importance.png",
    "xgboost_importance_bar.png",
    "xgboost_feature_importance.csv",
    "xgboost_explanation_true_positive.png",
    "xgboost_explanation_false_positive.png",
    "xgboost_dependence_plots.png",
    "cnn_global_importance.png",
    "cnn_importance_bar.png",
    "cnn_feature_importance.csv",
    "xgb_vs_cnn_feature_comparison.csv",
    "xgb_vs_cnn_comparison.png"
]

for f in files:
    print(f"   • {f}")

print(f"\n   All saved to: {SHAP_DIR}")

print("\n" + "=" * 70)
print("KEY INSIGHTS")
print("=" * 70)

insights = [
    "Both models now have explainable predictions",
    f"Feature agreement correlation: {spearman_corr:.2f}",
    "Individual predictions can be explained to analysts",
    "Dashboard-ready explanation function created",
    "Complementary feature priorities justify ensemble approach"
]

for insight in insights:
    print(f"   • {insight}")

print("\n" + "=" * 70)
print("NEXT STEPS")
print("=" * 70)
print("   1. Create model ensemble (XGBoost + CNN)")
print("   2. Integrate SHAP into dashboard")
print("   3. Test with SOC analysts")
print("   4. Final presentation preparation")

print("\n" + "=" * 70)
print("✨ SHAP EXPLAINABILITY COMPLETE! ✨")
print("=" * 70 + "\n")